In [7]:
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from ml_utils.datasets import LabelDataset
from torch import nn
from torch.utils.data import DataLoader
from sklearn.metrics import r2_score

SEED = 50

# 波士顿房价预测训练,数据来源https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/data
train_ori_data = pd.read_csv('../data/house-prices/train.csv')

all_features = train_ori_data.iloc[:,1:-1]

all_label = train_ori_data.iloc[:,-1:]

# 在特征中筛选出数字列，把空值用均值代替并且做归一化
number_columns = all_features.dtypes[all_features.dtypes != 'object'].index
# 用每列均值填充数值缺失
all_features[number_columns] = all_features[number_columns].fillna(all_features[number_columns].mean())
# Z-score 标准化,不同量纲的特征（如“年龄”和“收入”）被缩放到可比的尺度
all_features[number_columns] = all_features[number_columns].apply(lambda x: (x - x.mean())/(x.std()))
# 类别特征独热编码
# 目的：把字符串类别变成 0/1 特征；保留 float（不要整表转 int，否则会破坏数值列的小数）。
all_features = pd.get_dummies(all_features, dummy_na=True)


# 拆回训练/测试 + 提取标签
# 目的：把合并后的特征再分回去；同时拿出训练集标签 SalePrice。
n_train = all_features.shape[0]   #shape返回数据的格式为（行数,列数）这里取0代表获取训练数据的行数
train_features = all_features.iloc[:n_train].values.astype(np.float32)   # 提取训练数据
train_labels_raw = all_label['SalePrice'].values.astype(np.float32) # 提取训练标签


X_train, X_valid, y_train_raw, y_valid_raw = train_test_split(train_features, train_labels_raw, test_size=0.2, random_state=SEED)

# 转换为tensor
features_train_tensor = torch.from_numpy(X_train)
features_test_tensor = torch.from_numpy(y_train_raw)
price_train_tensor = torch.from_numpy(X_valid)
price_test_tensor = torch.from_numpy(y_valid_raw)

batch_size = 64

# 转换为Dataset
train_dataset = LabelDataset(features_train_tensor, features_test_tensor)
test_dataset = LabelDataset(price_train_tensor,price_test_tensor)

# Create data loaders.
train_dataloader = DataLoader(train_dataset, batch_size=batch_size)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size)


# 正确的设备选择
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()
    else "cpu"
)

print(f"使用 {device} 设备")

class HousePricePredictor(nn.Module):
    def __init__(self, n_features: int = 10):
        super().__init__()
        # 输入层 → 隐藏层1
        self.layer1 = nn.Linear(n_features, 128)
        # 隐藏层1 → 隐藏层2
        self.layer2 = nn.Linear(128, 64)
        # 隐藏层2 → 输出层（房价是1个连续值）
        self.output_layer = nn.Linear(64, 1)
        # 可选：加入 Dropout 防止过拟合（尤其数据少时）
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        x = torch.relu(self.layer1(x))
        x = self.dropout(x)
        x = torch.relu(self.layer2(x))
        x = self.dropout(x)
        x = self.output_layer(x)  # 注意：回归任务最后不要加激活函数(sigmoid)！
        return x.squeeze(-1)  # 去掉最后一个维度，返回 (batch_size,) 而不是 (batch_size, 1)



model = HousePricePredictor(n_features=330).to(device)      # 创建模型实例并移动到设备
print(features_train_tensor.shape)

loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

def train(dataloader, model, loss_fn, optimizer):
    """
    :param dataloader: 提供批量数据（如 DataLoader 实例）
    :param model: 要训练的神经网络；
    :param loss_fn: 损失函数
    :param optimizer: 优化器。
    :return:
    """
    size = len(dataloader.dataset)      # 获取整个训练集样本总数（用于打印进度）。
    model.train()       # 非常重要！ 将模型切换到 训练模式（training mode）。 影响某些层的行为，例如： Dropout：训练时启用，推理时关闭； BatchNorm：训练时用 batch 统计量，推理时用全局统计量。 如果不调用 .train()，模型可能表现异常（尤其用了这些层时）。
    for batch, (X, y) in enumerate(dataloader):   #  每次返回一个 batch 的 (X, y)，X：输入图像。y：标签
        X, y = X.to(device), y.to(device)
        pred = model(X)     # 调用模型的 forward() 方法，得到 logits（shape [64, 10]），向前传播
        loss = loss_fn(pred, y)     # 计算当前 batch 的平均损失（标量）。
        # 反向传播更新参数
        loss.backward()     # 自动计算损失对所有可学习参数的梯度（通过反向传播算法）。 梯度会累加到每个参数的 .grad 属性中。
        optimizer.step()        # 根据当前梯度（.grad）和优化器规则（如 SGD: w = w - lr * grad）更新参数。
        optimizer.zero_grad()       #  清空梯度缓冲区（将所有 .grad 设为 None 或 0）。 如果不清零，梯度会累积！ 下一次 backward() 会把新梯度加到旧梯度上，导致错误更新。
        loss, current = loss.item(), (batch + 1) * len(X)
        print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


def test(dataloader, model, loss_fn):
    model.eval()
    test_loss = 0.0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            all_preds.append(pred.cpu().numpy())
            all_targets.append(y.cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)
    test_loss /= len(dataloader)

    # 计算 MAE
    mae = np.mean(np.abs(all_preds - all_targets))
    # 计算 R²
    r2 = r2_score(all_targets, all_preds)

    print(f"测试结果:\n"
          f"  MSE:  {test_loss:>8f}\n"
          f"  MAE:  {mae:>8f}\n"
          f"  R²:   {r2:>8f}\n")
    return test_loss, mae, r2


epochs = 100
for t in range(epochs):     # 类似于java的 for(int i = 0;i<10;i++)
    print(f"回合 {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("完成!")

torch.save(model.state_dict(), "../data/house-prices/model.pth")
print("保存了训练模型到../data/house-prices/model.pth")

使用 cuda 设备
torch.Size([1168, 330])
回合 1
-------------------------------
loss: 40980402176.000000  [   64/ 1168]
loss: 37722431488.000000  [  128/ 1168]
loss: 41527320576.000000  [  192/ 1168]
loss: 46694154240.000000  [  256/ 1168]
loss: 36389793792.000000  [  320/ 1168]
loss: 31031193600.000000  [  384/ 1168]
loss: 41475661824.000000  [  448/ 1168]
loss: 32473063424.000000  [  512/ 1168]
loss: 39719911424.000000  [  576/ 1168]
loss: 38797357056.000000  [  640/ 1168]
loss: 36514856960.000000  [  704/ 1168]
loss: 26612316160.000000  [  768/ 1168]
loss: 57685860352.000000  [  832/ 1168]
loss: 43663241216.000000  [  896/ 1168]
loss: 39699750912.000000  [  960/ 1168]
loss: 42424946688.000000  [ 1024/ 1168]
loss: 32587763712.000000  [ 1088/ 1168]
loss: 33610340352.000000  [ 1152/ 1168]
loss: 42949398528.000000  [  304/ 1168]
测试结果:
  MSE:  39312780492.800003
  MAE:  181641.453125
  R²:   -5.070328

回合 2
-------------------------------
loss: 40978968576.000000  [   64/ 1168]
loss: 37720875008